PLEASE REFER TO APPENDIX ON HOW TO RUN THE CODE, CHANGE THE FILE NAMES ACCORDINGLY IN ORDER FOR THE DATASETS TO BE IMPORTED ON YOUR SYSTEM. YOU CAN LOCATE THE WHEREABOUTS OF THE CSV'S BY CHECKING THE PATHWAY WHERE THE CODE IS TRYING TO IMPORT


USE localized_url function if there's an error with the imports

In [ ]:
import os
def localized_url(url):

    current_dir = os.getcwd().split("statistical-analysis-and-modeling-of-football-data")[0] + "statistical-analysis-and-modeling-of-football-data"
    branch = url.split("statistical-analysis-and-modeling-of-football-data")[1]
    return current_dir + branch

In [ ]:
import requests
import json
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
#ignore warnings
import warnings
warnings.filterwarnings("ignore")
import ScraperFC as sfc
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from tqdm import tqdm

In [8]:

def get_json(id):
    options = Options()
    options.add_argument("--headless")  # Optional: Run without opening browser
    driver = webdriver.Chrome(options=options)
    url = f"https://www.sofascore.com/api/v1/event/{id}/statistics"
    # Load the Sofascore match page
    driver.get(url)
    driver.implicitly_wait(5)

    # Get full HTML content
    html = driver.page_source
    driver.quit()

    # Parse with BeautifulSoup
    soup = BeautifulSoup(html, "html.parser")

    # Extract the content inside the <pre> tag (contains your JSON)
    pre_tag = soup.find("pre")

    # Convert the JSON string to a Python dictionary
    if pre_tag:
        data = json.loads(pre_tag.text)
        return data
    else:
        print("❌ No <pre> tag found. Page structure may have changed.")




In [9]:
def get_stats(id):
    url = f"https://www.sofascore.com/api/v1/event/{id}/statistics"
    try:
        # Make the request
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad responses
    except requests.exceptions.RequestException as e:
        data = get_json(id)
    else:
         data = response.json()
   

    # Extract statistics groups safely
    stats = data.get('statistics', [{}])[0].get('groups', [])
    
    # Build list of DataFrames (each stat group)
    stats_df_list = [
        pd.DataFrame(group.get('statisticsItems', []))[['name', 'home', 'away']].set_index('name').transpose()
        for group in stats[0:] if 'statisticsItems' in group
    ]

    # Return empty DataFrame if there's no data
    if not stats_df_list:
        return pd.DataFrame()

    # Merge all groups into one row-wise stats frame (home and away are rows, stats are columns)
    final_df = pd.concat(stats_df_list, axis=1)

    # Optional: deduplicate column names in case of overlap
    final_df = final_df.loc[:, ~final_df.columns.duplicated()]

    # Reset index to turn "home"/"away" into a column
    final_df.reset_index(inplace=True)
    final_df.rename(columns={'index': 'team'}, inplace=True)

     # Separate home and away into a single row with suffixes
    home_stats = final_df[final_df['team'] == 'home'].drop(columns='team').add_suffix('_home')
    away_stats = final_df[final_df['team'] == 'away'].drop(columns='team').add_suffix('_away')

    combined_df = pd.concat([home_stats.reset_index(drop=True), away_stats.reset_index(drop=True)], axis=1)
    
    return combined_df

In [10]:
stats_df = get_stats(11352485);stats_df.columns


Index(['Ball possession_home', 'Expected goals_home', 'Big chances_home',
       'Total shots_home', 'Goalkeeper saves_home', 'Corner kicks_home',
       'Fouls_home', 'Passes_home', 'Tackles_home', 'Free kicks_home',
       'Yellow cards_home', 'Shots on target_home', 'Hit woodwork_home',
       'Shots off target_home', 'Blocked shots_home', 'Shots inside box_home',
       'Shots outside box_home', 'Big chances missed_home',
       'Fouled in final third_home', 'Offsides_home', 'Accurate passes_home',
       'Throw-ins_home', 'Final third entries_home', 'Long balls_home',
       'Crosses_home', 'Duels_home', 'Dispossessed_home', 'Ground duels_home',
       'Aerial duels_home', 'Dribbles_home', 'Tackles won_home',
       'Total tackles_home', 'Interceptions_home', 'Recoveries_home',
       'Clearances_home', 'Total saves_home', 'Goals prevented_home',
       'Goal kicks_home', 'Ball possession_away', 'Expected goals_away',
       'Big chances_away', 'Total shots_away', 'Goalkeeper save

In [25]:
sofa=sfc.Sofascore()
epl=sofa.get_match_dicts(league="EPL", year="22/23")


In [16]:
epl_23=pd.DataFrame()

In [42]:
epl[10]

{'tournament': {'name': 'Premier League',
  'slug': 'premier-league',
  'category': {'id': 1,
   'country': {'alpha2': 'EN',
    'alpha3': 'ENG',
    'name': 'England',
    'slug': 'england'},
   'name': 'England',
   'slug': 'england',
   'sport': {'name': 'Football', 'slug': 'football', 'id': 1},
   'flag': 'england',
   'alpha2': 'EN',
   'fieldTranslations': {'nameTranslation': {'ar': 'إنجلترا',
     'hi': 'इंग्लैंड',
     'bn': 'ইংল্যান্ড'},
    'shortNameTranslation': {}}},
  'uniqueTournament': {'name': 'Premier League',
   'slug': 'premier-league',
   'primaryColorHex': '#3c1c5a',
   'secondaryColorHex': '#f80158',
   'category': {'id': 1,
    'country': {'alpha2': 'EN',
     'alpha3': 'ENG',
     'name': 'England',
     'slug': 'england'},
    'name': 'England',
    'slug': 'england',
    'sport': {'name': 'Football', 'slug': 'football', 'id': 1},
    'flag': 'england',
    'alpha2': 'EN',
    'fieldTranslations': {'nameTranslation': {'ar': 'إنجلترا',
      'hi': 'इंग्लैंड',
 

'season': {'name': 'Premier League 22/23', 'year': '22/23', 'editor': False, 'seasonCoverageInfo': {}, 'id': 41886}, 'roundInfo': {'round': 37}, 'customId': 'GO', 'status': {'code': 60, 'description': 'Postponed', 'type': 'postponed'}, 'homeTeam': {'id': 39, 'country': {'alpha2': 'EN', 'alpha3': 'ENG', 'name': 'England', 'slug': 'england'}, 'name': 'Newcastle United', 'slug': 'newcastle-united', 'shortName': 'Newcastle', 'gender': 'M', 'sport': {'name': 'Football', 'slug': 'football', 'id': 1}, 'userCount': 940274, 'nameCode': 'NEW', 'disabled': False, 'national': False, 'type': 0, 'entityType': 'team', 'subTeams': [], 'teamColors': {'primary': '#000000', 'secondary': '#ffffff', 'text': '#ffffff'}, 'fieldTranslations': {'nameTranslation': {'ar': 'نيوكاسل يونايتد', 'ru': 'Ньюкасл Юнайтед', 'hi': 'न्यूकैसल यूनाइटेड', 'bn': 'নিউক্যাসল ইউনাইটেড'}, 'shortNameTranslation': {'ar': 'نيوكاسل', 'hi': 'न्यूकैसल', 'bn': 'নিউক্যাসল'}}}, 'awayTeam': {'id': 31, 'country': {'alpha2': 'EN', 'alpha3': 'ENG', 'name': 'England', 'slug': 'england'}, 'name': 'Leicester City', 'slug': 'leicester-city', 'shortName': 'Leicester', 'gender': 'M', 'sport': {'name': 'Football', 'slug': 'football', 'id': 1}, 'userCount': 546495, 'nameCode': 'LEI', 'disabled': False, 'national': False, 'type': 0, 'entityType': 'team', 'subTeams': []

In [ ]:
date = [datetime.utcfromtimestamp(epl[i]["startTimestamp"]).strftime("%Y-%m-%d %H:%M:%S") for i in range(len(epl))];epl_23['date'] = date
id = [epl[i]["id"] for i in range(len(epl))];epl_23['id'] = id

home_team = [epl[i]["homeTeam"]["name"] for i in range(len(epl))];epl_23['home_team'] = home_team
away_team = [epl[i]["awayTeam"]["name"] for i in range(len(epl))];epl_23['away_team'] = away_team
home_score = [epl[i]["homeScore"]["current"] for i in range(len(epl))];epl_23['home_score'] = home_score
away_score = [epl[i]["awayScore"]["current"] for i in range(len(epl))];epl_23['away_score'] = away_score

KeyError: 'current'

In [14]:
epl_23.head(10)

,date,id,home_team,away_team,home_score,away_score
0,2024-05-04 14:00:00,11352485,Brentford,Fulham,0,0
1,2024-05-04 14:00:00,11352491,Burnley,Newcastle United,1,4
2,2024-05-04 16:30:00,11352507,Manchester City,Wolverhampton,5,1
3,2024-05-05 13:00:00,11352488,Brighton & Hove Albion,Aston Villa,1,0
4,2024-05-05 13:00:00,11352494,Chelsea,West Ham United,5,0
5,2024-05-05 15:30:00,11352500,Liverpool,Tottenham Hotspur,4,2
6,2024-05-06 19:00:00,12226495,Crystal Palace,Manchester United,4,0
7,2024-05-11 11:30:00,11352529,Fulham,Manchester City,0,4
8,2024-05-11 14:00:00,11352519,Bournemouth,Brentford,1,2
9,2024-05-11 14:00:00,11352527,Everton,Sheffield United,1,0


In [24]:
epl_23.to_csv("indexes_2324.csv", index=False)

In [25]:
id_list = epl_23['id'].tolist()
epl_stats=pd.DataFrame()
for i in range(len(id_list)):
    try:
        stats_df = get_stats(id_list[i])
        stats_df['id'] = id_list[i]
        epl_stats = pd.concat([epl_stats, stats_df], ignore_index=True)
    except Exception as e:
        print(f"Error processing ID {id_list[i]}: {e}")


In [26]:
epl_stats_copy = epl_stats.copy()

In [27]:
epl_stats_copy.to_csv("stats_epl2324_unprocessed.csv", index=False)

In [55]:

def stats_to_dataframe(year, league):
    """
    Fetch match metadata and detailed stats from Sofascore API.

    Parameters:
        year (int): Season year
        league (str): League identifier used by Sofascore

    Returns:
        season_stats (DataFrame): Match metadata (date, teams, scores, ids)
        season_stats_final (DataFrame): Match statistics with ID reference
    """
    sofa = sfc.Sofascore()
    season_indeces = sofa.get_match_dicts(league=league, year=year)

    valid_matches = []

    for match in season_indeces:
        try:
            # Skip matches with no score data (e.g., postponed)
            if (
                "homeScore" not in match or "current" not in match["homeScore"] or
                "awayScore" not in match or "current" not in match["awayScore"]
            ):
                continue

            valid_matches.append({
                'date': datetime.utcfromtimestamp(match["startTimestamp"]).strftime("%Y-%m-%d %H:%M:%S"),
                'id': match["id"],
                'home_team': match["homeTeam"]["name"],
                'away_team': match["awayTeam"]["name"],
                'home_score': match["homeScore"]["current"],
                'away_score': match["awayScore"]["current"]
            })

        except Exception as e:
            print(f"Skipping invalid match: {e}")
            continue

    # Convert list to DataFrame
    season_stats = pd.DataFrame(valid_matches)
    season_stats_final = pd.DataFrame()

    # Fetch detailed stats for each valid match
    for match_id in tqdm(season_stats['id'].tolist(), desc="Fetching match stats"):
        try:
            stats_df = get_stats(match_id)
            stats_df['id'] = match_id
            season_stats_final = pd.concat([season_stats_final, stats_df], ignore_index=True)
        except Exception as e:
            print(f"❌ Error processing ID {match_id}: {e}")

    return season_stats, season_stats_final


In [46]:
test_indexes, test_stats = stats_to_dataframe(year="22/23", league="EPL")

Fetching match stats: 100%|██████████| 380/380 [05:07<00:00,  1.23it/s]


In [49]:
pd.DataFrame(test_indexes)

,date,id,home_team,away_team,home_score,away_score
0,2023-05-13 14:00:00,10385697,Chelsea,Nottingham Forest,2,2
1,2023-05-13 14:00:00,10385702,Manchester United,Wolverhampton,2,0
2,2023-05-14 13:00:00,10385688,Brentford,West Ham United,2,0
3,2023-05-14 13:00:00,10385685,Everton,Manchester City,0,3
4,2023-05-14 15:30:00,10385684,Arsenal,Brighton & Hove Albion,0,3
...,...,...,...,...,...,...
375,2022-08-07 13:00:00,10385491,Leicester City,Brentford,2,2
376,2022-08-07 13:00:00,10385511,Manchester United,Brighton & Hove Albion,1,2
377,2022-08-07 15:30:00,10385514,West Ham United,Manchester City,0,2
378,2022-08-13 11:30:00,10385503,Aston Villa,Everton,2,1


In [56]:
from datetime import datetime
def stats_to_dataframe_range(start_season, end_season, league):
    """
    Fetch match metadata and stats from Sofascore across a range of football seasons.
    
    Parameters:
        start_season (str): Starting season in format "YY/YY" (e.g., "15/16")
        end_season (str): Ending season in format "YY/YY" (e.g., "23/24")
        league (str): League identifier used by Sofascore

    Returns:
        combined_meta_df (DataFrame): All match metadata across seasons
        combined_stats_df (DataFrame): All match stats across seasons
    """



    all_meta = []
    all_stats = []

    # Convert to numeric range
    start = int(start_season.split('/')[0])
    end = int(end_season.split('/')[0]) + 1

    for year in range(start, end):
        season_str = f"{year:02d}/{(year + 1) % 100:02d}"
        print(f"🔄 Processing season {season_str}...")

        retries = 3
        attempt = 0
        while attempt < retries:
            try:
                meta_df, stats_df = stats_to_dataframe(year=season_str, league=league)
                meta_df['season'] = season_str
                stats_df['season'] = season_str
                all_meta.append(meta_df)
                all_stats.append(stats_df)
                print(f"✅ Successfully processed season {season_str}")
                break  # Exit retry loop if successful
            except Exception as e:
                attempt += 1
                print(f"⚠️ Attempt {attempt}/{retries} failed for {season_str}: {e}")
                if attempt == retries:
                    print(f"❌ Failed to process season {season_str} after {retries} attempts. Moving on...")

    combined_meta_df = pd.concat(all_meta, ignore_index=True)
    combined_stats_df = pd.concat(all_stats, ignore_index=True)

    return combined_meta_df, combined_stats_df

In [57]:
meta_all, stats_all = stats_to_dataframe_range("15/16", "23/24", league="EPL")

🔄 Processing season 15/16...


Fetching match stats: 100%|██████████| 380/380 [05:30<00:00,  1.15it/s]


✅ Successfully processed season 15/16
🔄 Processing season 16/17...


Fetching match stats: 100%|██████████| 380/380 [05:39<00:00,  1.12it/s]


✅ Successfully processed season 16/17
🔄 Processing season 17/18...


...

✅ Successfully processed season 21/22
🔄 Processing season 22/23...


Fetching match stats: 100%|██████████| 380/380 [04:53<00:00,  1.29it/s]


✅ Successfully processed season 22/23
🔄 Processing season 23/24...


Fetching match stats: 100%|██████████| 380/380 [05:14<00:00,  1.21it/s]

✅ Successfully processed season 23/24


In [58]:
meta_all.to_csv("indexes_1524.csv", index=False)
stats_all.to_csv("stats_epl1524.csv", index=False)